In [79]:
import numpy as np
import datetime
import os
from netCDF4 import Dataset
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import itertools
np.float = float
import cartopy
from regrid import regrid
import tqdm
import warnings
warnings.filterwarnings("ignore")

In [80]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

In [81]:
region_mask = np.array(Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')['mask'])

In [82]:
region_mask[ease_lats<55] = 0
region_mask[ease_lats>62] = 0
region_mask[ease_lons>-76] = 0

In [83]:
cs2_path = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2/MERRA2/'
smos_path = '/Users/carmennab/Dropbox/pb_new/data/SMOS/'
datapath_out = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/'

In [84]:
d1 = datetime.date(2010, 1, 1)
d2 = datetime.date(2022, 12, 31)
dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]
str_dates = [date.strftime('%Y%m%d') for date in dates]

In [ ]:
for date,str_date in zip(dates,str_dates):

    ### check if SMOS data & grid on EASE
    smos_filepath = f'{smos_path}{date.year}/{str_date[4:6]}/SMOS_Icethickness_v3.3_north_{str_date}.nc'
    cs2_filepath = f'{cs2_path}{date.year}/{str_date}.nc'

    if os.path.exists(smos_filepath) == True: 
        smos_file = Dataset(smos_filepath)
        smos_lons = np.array(smos_file['longitude'])
        smos_lats = np.array(smos_file['latitude'])
        smos_sit = np.array(smos_file['sea_ice_thickness'])[0]
        sit_unc = np.array(smos_file['ice_thickness_uncertainty'])[0]
        smos_sit[sit_unc>1]=np.nan
        smos_sit[smos_sit<0]=np.nan
        smos_sit[smos_sit>1]=np.nan

        smos_ease = regrid(smos_sit,smos_lons,smos_lats,ease_lons,ease_lats)
        combined_sit = smos_ease.copy()

        ### check if CS2 data
        if os.path.exists(cs2_filepath) == True:    
            cs2_sit = np.array(Dataset(cs2_filepath)['sea_ice_thickness'])
            cs2_sit[cs2_sit<0] = np.nan
            sd = np.array(Dataset(cs2_filepath)['snow_depth'])

            ### combine SMOS and CS2
            for i, j in itertools.product(np.arange(ease_lons.shape[0]),np.arange(ease_lons.shape[1])):
                if np.isnan(combined_sit[i,j]):
                    combined_sit[i,j] = cs2_sit[i,j]
        else:
            sd = np.full((84,103),np.nan)

        combined_sit[~np.isin(region_mask,4)] = np.nan # remove non-HB data
        if ~np.isnan(combined_sit).all():
            filepath_out = f'{datapath_out}/{date.year}/{str_date}.nc'
            if os.path.exists(filepath_out) == True: 
                ! rm {filepath_out}
            data_vars = {'sea_ice_thickness':(['x','y'],combined_sit),
                         'snow_depth':(['x','y'],sd)}
            ds = xr.Dataset(data_vars = data_vars,
                            coords={'longitude':(['x','y'],ease_lons),
                                    'latitude':(['x','y'],ease_lats)})
            ds.to_netcdf(filepath_out,'w')

    ### if no SMOS, just use CS2 where available
    elif os.path.exists(cs2_filepath) == True:    
        combined_sit = np.array(Dataset(cs2_filepath)['sea_ice_thickness'])
        combined_sit[combined_sit<0] = np.nan
        sd = np.array(Dataset(cs2_filepath)['snow_depth'])

        combined_sit[~np.isin(region_mask,4)] = np.nan # remove non-HB data
        if ~np.isnan(combined_sit).all():
            filepath_out = f'{datapath_out}/{date.year}/{str_date}.nc'
            if os.path.exists(filepath_out) == True: 
                ! rm {filepath_out}
            data_vars = {'sea_ice_thickness':(['x','y'],combined_sit),
                         'snow_depth':(['x','y'],sd)}
            ds = xr.Dataset(data_vars = data_vars,
                            coords={'longitude':(['x','y'],ease_lons),
                                    'latitude':(['x','y'],ease_lats)})
            ds.to_netcdf(filepath_out,'w')